# 🎵 MusicGen Fine-Tuning Journey
## Notebook 2: Data Preparation

---

### What we're doing in this notebook

Fine-tuning a model requires *paired data*: audio clips + text descriptions of those clips. In this notebook we:

1. **Download** throat singing audio from YouTube using `yt-dlp`
2. **Convert** everything to the format MusicGen expects (mono WAV, 32kHz)
3. **Segment** long recordings into fixed-length clips (10 seconds)
4. **Write text captions** that describe each clip
5. **Save a manifest** file that the training notebook will read

### How much data do we need?

This is a genuinely interesting question for domain adaptation:
- **Too little (< 10 clips)**: The model will memorize the clips rather than learn the style
- **Sweet spot (~50-200 clips)**: Enough to generalize the style, manageable to collect
- **More is always better**, but returns diminish quickly for style adaptation

Each YouTube video of a live concert might give you 10-30 usable 10-second segments. So ~10-20 videos = ~100-300 clips = a solid small dataset.

### A note on copyright

Downloading YouTube audio for a **personal research/learning project** (not commercial, not redistributed) is widely treated as acceptable for ML research. You should not redistribute your dataset or the fine-tuned model for commercial purposes. Traditional/folk music with no clear copyright ownership (much pre-modern Tuvan/Mongolian music) is the safest zone.


## Step 1: Install Data Collection Tools

In [ ]:
# yt-dlp: modern YouTube downloader (ffmpeg-based)
# pydub: audio manipulation
# soundfile: read/write WAV
!pip install yt-dlp pydub soundfile tqdm --quiet

# ffmpeg is required by yt-dlp for audio conversion
# Check if it's available:
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    version_line = result.stdout.split('\n')[0]
    print(f"✅ ffmpeg found: {version_line}")
else:
    print("❌ ffmpeg not found! Install it:")
    print("   WSL2/Ubuntu: sudo apt-get install ffmpeg")
    print("   Then restart this kernel.")

In [ ]:
# All imports for this notebook
import os
import json
import subprocess
import shutil
from pathlib import Path

import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import IPython.display as ipd
from tqdm import tqdm

# ─── Dataset directories ─────────────────────────────────────────────
RAW_DIR      = Path("dataset/raw")        # Downloaded audio (before processing)
PROCESSED_DIR = Path("dataset/processed") # Segmented 10s WAV clips
DATASET_DIR  = Path("dataset")            # Root

for d in [RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# MusicGen's expected sample rate
TARGET_SR = 32000
CLIP_DURATION = 10.0  # seconds per clip
CLIP_SAMPLES = int(TARGET_SR * CLIP_DURATION)

print("✅ Setup complete")
print(f"   Raw downloads:   {RAW_DIR}")
print(f"   Processed clips: {PROCESSED_DIR}")
print(f"   Target SR:       {TARGET_SR} Hz")
print(f"   Clip duration:   {CLIP_DURATION}s ({CLIP_SAMPLES} samples)")

## Step 2: Define Your Sources

Below is a curated list of YouTube search queries and specific artists known for high-quality traditional throat singing. We'll use `yt-dlp`'s `ytsearch` feature to find and download the top results for each query.

**The main styles we want to capture:**
- **Khoomei** (хоомей): the base Tuvan throat singing technique — harmonic overtones over a drone
- **Sygyt** (сыгыт): produces a clear, flute-like high overtone melody  
- **Kargyraa** (каргыраа): a very deep, gravelly subharmonic drone
- **Mongolian urtiin duu**: long songs with throat-like extended vocal techniques

Having **variety in style** helps the model learn the category rather than one specific song.

> **You can add your own YouTube URLs** to the `SPECIFIC_URLS` list below — any URL `yt-dlp` supports works.

In [ ]:
# ─── Search Queries ───────────────────────────────────────────────────
# Each entry: (search_query, n_results, style_tag)
# yt-dlp will download the top n_results for each query

SEARCH_QUERIES = [
    # (query, max_results_to_download, style)
    ("Huun Huur Tu throat singing traditional tuvan",     3, "khoomei"),
    ("tuvan throat singing khoomei concert live",          3, "khoomei"),
    ("sygyt throat singing tuvan flute overtone",          3, "sygyt"),
    ("kargyraa tuvan throat singing deep",                 3, "kargyraa"),
    ("mongolian throat singing traditional folk",          3, "mongolian"),
    ("Tanya Tagaq throat singing",                         2, "katajjaq"),  # Inuit variant
    ("Sainkho Namtchylak tuvan singing",                   2, "experimental"),
]

# ─── Specific URLs you trust (add any YouTube links here) ─────────────
# Format: (url, style_tag)
SPECIFIC_URLS = [
    # Add your own here, e.g.:
    # ("https://www.youtube.com/watch?v=XXXXXXX", "khoomei"),
]

print(f"Search queries: {len(SEARCH_QUERIES)}")
max_from_search = sum(n for _, n, _ in SEARCH_QUERIES)
print(f"Max videos from search: {max_from_search}")
print(f"Specific URLs: {len(SPECIFIC_URLS)}")
print(f"Expected ~{(max_from_search + len(SPECIFIC_URLS)) * 5}-{(max_from_search + len(SPECIFIC_URLS)) * 15} clips after segmentation")

## Step 3: Download Audio

`yt-dlp` can download audio-only and convert directly to WAV. The `ytsearch{N}:query` syntax tells it to search YouTube and return the top N results.

We download at the original quality and resample later (always better than asking yt-dlp to resample to an unusual rate).

In [ ]:
def download_audio(url_or_search: str, output_dir: Path, style_tag: str) -> list:
    """
    Download audio from a URL or yt-dlp search query.
    Returns list of downloaded file paths.
    """
    output_template = str(output_dir / f"{style_tag}_%(title).50s_%(id)s.%(ext)s")

    cmd = [
        'yt-dlp',
        '-x',                          # extract audio only
        '--audio-format', 'wav',       # convert to WAV
        '--audio-quality', '0',        # best quality
        '--no-playlist',               # don't download whole playlists
        '--no-warnings',
        '--quiet',
        '-o', output_template,
        url_or_search
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 and result.stderr:
        print(f"   ⚠️  Warning: {result.stderr[:200]}")

    # Find newly created WAV files
    return list(output_dir.glob(f"{style_tag}_*.wav"))


# ─── Download from search queries ────────────────────────────────────
all_downloaded = []

print("Downloading from search queries...")
print("(This may take several minutes depending on connection speed)\n")

for query, n_results, style in SEARCH_QUERIES:
    search_str = f"ytsearch{n_results}:{query}"
    print(f"  [{style}] {query[:50]}...")

    files = download_audio(search_str, RAW_DIR, style)
    new_files = [f for f in files if f not in all_downloaded]
    all_downloaded.extend(new_files)
    print(f"         → {len(new_files)} file(s) downloaded")

# ─── Download specific URLs if any ───────────────────────────────────
if SPECIFIC_URLS:
    print("\nDownloading specific URLs...")
    for url, style in SPECIFIC_URLS:
        print(f"  [{style}] {url}")
        files = download_audio(url, RAW_DIR, style)
        all_downloaded.extend(files)

print(f"\n✅ Total downloaded: {len(all_downloaded)} file(s)")
for f in all_downloaded:
    size_mb = f.stat().st_size / 1e6
    print(f"   {f.name} ({size_mb:.1f} MB)")

## Step 4: Inspect Raw Downloads

Before processing, let's look at what we actually got — duration, sample rate, channels. This helps us understand what preprocessing is needed.

In [ ]:
# Scan all downloaded WAVs and report properties
raw_files = sorted(RAW_DIR.glob('*.wav'))

if not raw_files:
    print("❌ No WAV files found in raw directory!")
    print("   Check that downloads succeeded above.")
else:
    print(f"{'File':<55} {'SR':>7} {'Dur (s)':>8} {'Ch':>4}")
    print('─' * 78)
    total_duration = 0
    for f in raw_files:
        try:
            info = sf.info(str(f))
            total_duration += info.duration
            print(f"{f.name[:54]:<55} {info.samplerate:>7} {info.duration:>8.1f} {info.channels:>4}")
        except Exception as e:
            print(f"{f.name[:54]:<55} ERROR: {e}")

    print('─' * 78)
    print(f"Total duration: {total_duration/60:.1f} minutes across {len(raw_files)} files")
    expected_clips = int(total_duration / CLIP_DURATION)
    print(f"Expected clips after {CLIP_DURATION}s segmentation: ~{expected_clips}")

## Step 5: Process — Resample, Mono-ify, Segment

We need to:
1. **Resample** to 32,000 Hz (MusicGen's sample rate)
2. **Convert to mono** (1 channel) — MusicGen processes mono audio
3. **Segment** into 10-second chunks
4. **Skip silent clips** — clips that are mostly silence add noise to training

Why 10 seconds? MusicGen-small can generate up to 30s, but 10s clips:
- Are long enough to capture the texture/style
- Short enough that we get many clips per source video
- Fit comfortably in GPU memory during training

In [ ]:
def is_silent(audio: np.ndarray, threshold_db: float = -40.0) -> bool:
    """
    Return True if the clip is mostly silence.
    threshold_db: clips quieter than this (in dBFS) are considered silent.
    """
    if len(audio) == 0:
        return True
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < 1e-10:
        return True
    db = 20 * np.log10(rms)
    return db < threshold_db


def process_file(input_path: Path, output_dir: Path, clip_dur: float = 10.0,
                 target_sr: int = 32000, silence_threshold_db: float = -40.0) -> list:
    """
    Load one audio file, resample, convert to mono, split into clips.
    Returns list of output paths for successfully created clips.
    """
    clip_samples = int(target_sr * clip_dur)
    saved = []

    try:
        # Load and resample in one step with librosa
        audio, _ = librosa.load(str(input_path), sr=target_sr, mono=True)
    except Exception as e:
        print(f"     ❌ Failed to load {input_path.name}: {e}")
        return []

    # Split into non-overlapping clips
    n_clips = len(audio) // clip_samples
    stem = input_path.stem[:30]  # shortened filename for output naming

    for i in range(n_clips):
        start = i * clip_samples
        clip = audio[start: start + clip_samples]

        if is_silent(clip, silence_threshold_db):
            continue  # skip silence

        # Normalize to [-1, 1] to avoid clipping
        peak = np.max(np.abs(clip))
        if peak > 0:
            clip = clip / peak * 0.95

        out_path = output_dir / f"{stem}_clip{i:04d}.wav"
        sf.write(str(out_path), clip, target_sr, subtype='PCM_16')
        saved.append(out_path)

    return saved


print("Processing all raw files...")
all_clips = []

for raw_file in tqdm(raw_files, desc="Processing"):
    clips = process_file(raw_file, PROCESSED_DIR)
    all_clips.extend(clips)
    print(f"   {raw_file.name[:50]:50s} → {len(clips)} clips")

print(f"\n✅ Processing complete")
print(f"   Total clips: {len(all_clips)}")
print(f"   Total audio: {len(all_clips) * CLIP_DURATION / 60:.1f} minutes")
print(f"   Saved to:    {PROCESSED_DIR}")

## Step 6: Write Text Descriptions (Captions)

This is **more important than most people realize**. The model learns to associate text descriptions with audio patterns. If your descriptions are too generic, the model won't learn a precise association. If they're too specific, it won't generalize.

We'll assign descriptions based on the style tag we recorded during download, with some variation to avoid the model memorizing one specific phrase.

**Good description principles:**
- Mention the technique name (khoomei, sygyt, kargyraa)
- Mention cultural context (tuvan, mongolian, traditional)
- Mention sonic character (drone, overtone, meditative, ancient)
- Vary the phrasing — don't have 100 clips all with identical descriptions

In [ ]:
import random
random.seed(42)

# Description templates per style
# Multiple templates → random selection → variety in training data
DESCRIPTION_TEMPLATES = {
    "khoomei": [
        "tuvan throat singing, khoomei technique, harmonic overtones, meditative drone, traditional",
        "khoomei throat singing, tuvan folk music, deep drone with overtone melody, ancient",
        "traditional tuvan throat singing, khoomei, overtone harmonics over sustained drone",
        "tuvan overtone singing, khoomei style, low fundamental with high harmonic melody",
        "tuvan throat singing, khoomei, traditional instrument, meditative, world music",
    ],
    "sygyt": [
        "sygyt throat singing, tuvan, high pitched flute-like overtone, whistle-like harmonics",
        "tuvan sygyt, throat singing with clear high overtone melody, traditional, ancient",
        "sygyt style tuvan throat singing, bright high harmonic above drone, folk",
        "tuvan throat singing sygyt technique, melodic overtone, nature sounds, meditative",
    ],
    "kargyraa": [
        "kargyraa throat singing, very deep subharmonic drone, gravelly texture, tuvan",
        "tuvan kargyraa, extremely low guttural drone, subharmonic vibration, ancient folk",
        "throat singing kargyraa style, deep resonant bass drone, traditional tuvan music",
        "tuvan throat singing kargyraa, low earthly drone, meditative, ritual music",
    ],
    "mongolian": [
        "mongolian throat singing, traditional folk music, drone harmonics, steppe music",
        "mongolian khoomii, overtone singing, ancient traditional music, nomadic",
        "mongolian traditional vocal music, overtone harmonics, folk, central asian",
        "mongolian throat singing, khoomii technique, drone with melody, ethnic music",
    ],
    "katajjaq": [
        "inuit throat singing, katajjaq style, rhythmic, vocal game, traditional",
        "throat singing, rhythmic vocal breathing, traditional indigenous music, drone",
        "traditional throat singing, rhythmic overtone vocal, ethnic, ancient",
    ],
    "experimental": [
        "experimental throat singing, extended vocal technique, overtone, avant-garde",
        "contemporary throat singing, modern, drone, harmonic overtone, world music",
        "tuvan inspired throat singing, drone, overtone, world fusion music",
    ],
}

# Fallback for unrecognized tags
DEFAULT_DESCRIPTIONS = [
    "throat singing, overtone vocal technique, drone, traditional world music",
    "traditional throat singing, drone with harmonics, ethnic music",
    "overtone singing, throat singing technique, ancient, meditative",
]


def get_description(clip_path: Path) -> str:
    """Infer style from filename prefix and return a varied description."""
    name = clip_path.stem.lower()
    for style, templates in DESCRIPTION_TEMPLATES.items():
        if name.startswith(style):
            return random.choice(templates)
    return random.choice(DEFAULT_DESCRIPTIONS)


# Preview
print("Sample descriptions for first 10 clips:")
print('─' * 70)
processed_clips = sorted(PROCESSED_DIR.glob('*.wav'))
for clip in processed_clips[:10]:
    desc = get_description(clip)
    print(f"{clip.name[:30]:30s} → {desc[:50]}...")

## Step 7: Save the Dataset Manifest

We'll save a **JSONL** (JSON Lines) file where each line is one training example:
```json
{"path": "clip_0001.wav", "description": "tuvan throat singing, khoomei...", "duration": 10.0}
```

This is our training manifest — the fine-tuning notebook reads this file to know what to train on.

We'll also do a **train/val split** (90/10). The validation set lets us check if the model is actually learning vs. memorizing.

In [ ]:
import random
random.seed(42)

all_clips_sorted = sorted(PROCESSED_DIR.glob('*.wav'))

if not all_clips_sorted:
    print("❌ No processed clips found. Check the processing step above.")
else:
    # Build dataset records
    records = []
    for clip_path in all_clips_sorted:
        desc = get_description(clip_path)
        records.append({
            "path": clip_path.name,     # relative to PROCESSED_DIR
            "description": desc,
            "duration": CLIP_DURATION,
        })

    # Shuffle before splitting
    random.shuffle(records)

    # 90/10 train/val split
    n_val = max(1, int(len(records) * 0.1))
    val_records = records[:n_val]
    train_records = records[n_val:]

    # Save manifest files
    train_manifest = DATASET_DIR / 'train.jsonl'
    val_manifest   = DATASET_DIR / 'val.jsonl'

    with open(train_manifest, 'w') as f:
        for r in train_records:
            f.write(json.dumps(r) + '\n')

    with open(val_manifest, 'w') as f:
        for r in val_records:
            f.write(json.dumps(r) + '\n')

    # Also save the data directory path for notebook 3
    config = {
        "data_dir": str(PROCESSED_DIR.resolve()),
        "train_manifest": str(train_manifest.resolve()),
        "val_manifest": str(val_manifest.resolve()),
        "sample_rate": TARGET_SR,
        "clip_duration": CLIP_DURATION,
        "n_train": len(train_records),
        "n_val": len(val_records),
    }
    with open(DATASET_DIR / 'dataset_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    print(f"✅ Manifest saved!")
    print(f"   Train: {len(train_records)} clips → {train_manifest}")
    print(f"   Val:   {len(val_records)} clips  → {val_manifest}")
    print(f"   Config → {DATASET_DIR / 'dataset_config.json'}")
    print(f"\n   Sample train entry:")
    print(f"   {json.dumps(train_records[0], indent=4)}")

## Step 8: Explore Your Dataset

Always visualize your data before training. This catches obvious issues (corrupt files, weird spectrograms, mislabeled clips) and also just builds intuition for what you're working with.

In [ ]:
# Show spectrogram + audio for a random sample of clips
n_show = min(6, len(all_clips_sorted))
sample_clips = random.sample(all_clips_sorted, n_show)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, clip_path in enumerate(sample_clips):
    audio, _ = librosa.load(str(clip_path), sr=TARGET_SR)
    S = librosa.feature.melspectrogram(y=audio, sr=TARGET_SR, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)

    librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                              sr=TARGET_SR, ax=axes[i], fmax=8000)

    desc = get_description(clip_path)
    axes[i].set_title(f'{clip_path.stem[:25]}\n"{desc[:40]}..."', fontsize=8)

plt.suptitle('Dataset Sample — Mel Spectrograms', fontsize=13)
plt.tight_layout()
plt.savefig('dataset/spectrogram_sample.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🎵 Audio playback — listen to a few clips:")
for clip_path in sample_clips[:3]:
    audio, _ = librosa.load(str(clip_path), sr=TARGET_SR)
    print(f"\n{clip_path.name} — {get_description(clip_path)[:60]}")
    display(ipd.Audio(audio, rate=TARGET_SR))

In [ ]:
# Dataset statistics
style_counts = {}
for clip_path in all_clips_sorted:
    name = clip_path.stem.lower()
    style = 'unknown'
    for s in DESCRIPTION_TEMPLATES:
        if name.startswith(s):
            style = s
            break
    style_counts[style] = style_counts.get(style, 0) + 1

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
styles = list(style_counts.keys())
counts = [style_counts[s] for s in styles]
bars = ax.bar(styles, counts, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#795548'][:len(styles)])
ax.set_xlabel('Style')
ax.set_ylabel('Number of 10s clips')
ax.set_title('Dataset Composition')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(count),
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('dataset/dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDataset Summary:")
for style, count in sorted(style_counts.items(), key=lambda x: -x[1]):
    print(f"  {style:<15}: {count:>4} clips ({count * CLIP_DURATION / 60:.1f} min)")
print(f"  {'TOTAL':<15}: {sum(counts):>4} clips ({sum(counts) * CLIP_DURATION / 60:.1f} min)")

## Notebook 2 Summary

**What we built:**
- A pipeline that downloads, converts, and segments throat singing audio
- Varied text descriptions that match the audio style
- Train/val split manifest files ready for the training notebook

**Data quality questions to ask yourself:**
1. Do the spectrograms look like actual throat singing? (Look for that characteristic drone + overtone lines)
2. Do the clips sound clean? (Low background noise is important)
3. Is the dataset balanced across styles? (Ideally 3+ clips per style)
4. Are the descriptions specific enough? (Compare: "music" vs "tuvan throat singing, khoomei, overtone drone")

**If you have fewer than 30 clips:** Consider adding more YouTube searches or supplementing with your own recordings. At < 20 clips, fine-tuning often leads to memorization rather than style learning.

---

**➡️ Next: `03_finetuning.ipynb` — Training the model on your new data**